[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanjaynagi/anopheles-omics-training/blob/main/day-3/3-2-selection-atlas.ipynb)

# 3.2 The selection atlas: is your gene under selection?

**Theme:** Analysis | Tools & technology

In Module 3.1 we found that the *Cyp6aa1* duplication is almost fixed around Busia, and that
other variants near our genes are common. A variant that helps mosquitoes survive insecticides
should rise in frequency quickly, dragging nearby DNA with it: a **selective sweep**. If our
candidate genes sit inside a sweep in wild Busia mosquitoes, that is independent evidence that
something at the locus matters for survival.

In the MalariaGEN–PAMCA course you learned to compute an H12 genome-wide selection scan yourself
([W6-M3](https://anopheles-genomic-surveillance.github.io/workshop-6/module-3-gwss.html)).
Here we use the **Malaria Vector Selection Atlas**, which has already run H12, G123 and iHS scans
for every well-sampled cohort in *Ag3* and called the selection signals in each one. We learn how
the signals are called, how to read the website, and how to query the signal table for our genes.

## Learning objectives

At the end of this module you will be able to:

- Explain in a few sentences what H12, G123 and iHS measure and when each is useful.
- Describe how the atlas calls a signal (peak fit, ΔAIC) and what the focus, span1 and span2 intervals mean.
- Navigate the atlas website (cohort, chromosome and alert pages).
- Query the signal table for East African cohorts and find signals that overlap your candidate genes, converting between arm (2R, 2L) and compound contig (2RL) coordinates.
- Compare signals at a locus across cohorts, taxa and years.

## Setup

We only need `malariagen_data` for gene coordinates and contig lengths; the atlas itself is a
single table.

In [ ]:
%pip install -q --no-warn-conflicts malariagen_data==15.9.0

In [2]:
import os
os.environ["MGEN_SHOW_PROGRESS"] = "0"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import malariagen_data

pio.renderers.default = "notebook_connected+colab"
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

ag3 = malariagen_data.Ag3()

In [ ]:
# TODO(data): the atlas signal table (h12-signal-detection-all.csv, selection-atlas repo root)
# is not yet published at a public URL. Update this constant once it is (see data/README.md).
SIGNALS_URL = "https://raw.githubusercontent.com/anopheles-genomic-surveillance/selection-atlas/main/h12-signal-detection-all.csv"

ATLAS = "https://anopheles-genomic-surveillance.github.io/selection-atlas/agam"
BUSIA = "UG-E_Busia_gamb_2016_Q2"

In [4]:
# Replace with the genes you were assigned on Day 2 (see data/candidate-genes.tsv)
CANDIDATE_GENES = ["AGAP002862", "AGAP009193"]  # CYP6AA1, GSTE4

## Theory: three selection statistics in brief

All three statistics look for the footprint of a recent sweep: a stretch of genome where many
mosquitoes carry the same, or very similar, haplotypes because one haplotype rose in frequency
faster than recombination could break it up.

- **H12** [1] measures haplotype homozygosity in windows of a fixed number of SNPs, adding
  together the frequencies of the two most common haplotypes. It is high for hard sweeps (one
  haplotype) and for soft sweeps (a few haplotypes). It needs phased haplotypes.
  This is the statistic you computed in [W6-M3](https://anopheles-genomic-surveillance.github.io/workshop-6/module-3-gwss.html).
- **G123** [2] is the unphased equivalent: it works on multilocus genotypes, so it does not
  depend on phasing. It usually tells the same story as H12.
- **iHS** [3] compares how far haplotype homozygosity extends around the two alleles of each SNP.
  It is most sensitive to *incomplete* sweeps, where the favoured allele is at intermediate
  frequency, and less sensitive once the sweep is near fixation.

Each statistic depends on the window size (H12, G123) and on the sample size, and demography can
mimic sweeps (for example in inbred populations; see
[W7-M3](https://anopheles-genomic-surveillance.github.io/workshop-7/module-3-haplotype-clustering.html)).
A signal says "something here was selected recently"; it does not say which variant.

## How the atlas calls a signal

The atlas groups *Ag3* mosquitoes into **cohorts**: females of one taxon from one level-2
administrative unit (district) collected in one quarter (or year). Cohorts need at least 15
mosquitoes; larger ones are downsampled to 100. For each cohort the H12 and G123 window size is
**calibrated** on chromosome 3RL: it is the smallest window for which 95 % of windows have a
value below 0.08. Busia 2016 has 24 mosquitoes and an H12 window of 2,000 SNPs.

Signals are then called from the H12 scan only:

1. Outlier windows are removed with a Hampel filter.
2. Physical positions (bp) are converted to genetic positions (cM) with a recombination map, so
   peak widths are comparable along the genome.
3. In steps of 1 cM along each chromosome, two models are fitted to the H12 values: a flat line
   (no signal) and a **skewed exponential peak** with a centre, amplitude, decay (width), skew and
   baseline.
4. A signal is kept if the peak fits much better than the flat line, **ΔAIC = AIC(flat) − AIC(peak) > 1000**,
   and the highest H12 value in the peak is above 0.1.
5. Overlapping signals in a cohort are merged, keeping the one with the largest ΔAIC.

From the fitted peak the atlas reports three nested intervals around the centre, on each side
using that side's decay constant:

| Interval | Width | Use |
|---|---|---|
| `focus` | centre ± 0.25 × decay | the core, most likely to contain the selected variant |
| `span1` | centre ± 1 × decay | a conservative extent of the signal |
| `span2` | centre ± 2 × decay | the broad footprint; genes here may just be hitchhiking |

Each interval is given in cM (`focus_gstart`, …) and bp (`focus_pstart`, …). Other useful
columns: `pcenter` (fitted centre, bp), `stat_max` (highest observed H12 in the signal, a rough
guide to how complete the sweep is) and `delta_i` (ΔAIC, how confident we are there is a peak).

**Chromosome naming.** The atlas scans the compound contigs **2RL** (2R followed by 2L) and
**3RL** (3R followed by 3L) so that sweeps near the centromere are not cut in two. Positions on
2L and 3L are therefore shifted by the length of 2R or 3R. We deal with this below.

## Reading the website

The atlas website is at <https://anopheles-genomic-surveillance.github.io/selection-atlas/>
(choose *An. gambiae* complex). It has four kinds of page:

- **Cohort pages**, e.g. [Uganda / Busia / gambiae / 2016 / Q2](https://anopheles-genomic-surveillance.github.io/selection-atlas/agam/cohort/UG-E_Busia_gamb_2016_Q2.html):
  sample details, a map, and H12, G123 and iHS tracks for each chromosome with the called signals
  shaded and a gene track underneath. Hover to see values; zoom to a signal.
- **Chromosome pages**, e.g. [2RL](https://anopheles-genomic-surveillance.github.io/selection-atlas/agam/contig/2RL.html):
  every cohort's signals stacked along one chromosome, which shows loci under selection in many
  places at once.
- **Country pages**, e.g. [Uganda](https://anopheles-genomic-surveillance.github.io/selection-atlas/agam/country/UG.html):
  all cohorts from one country.
- **Selection alerts**: named loci with known resistance genes and the cohorts with signals there,
  e.g. [SA-AGAM-04 *Cyp6aa/Cyp6p*](https://anopheles-genomic-surveillance.github.io/selection-atlas/agam/alert/SA-AGAM-04.html),
  [SA-AGAM-05 *Cyp9k1*](https://anopheles-genomic-surveillance.github.io/selection-atlas/agam/alert/SA-AGAM-05.html)
  and [SA-AGAM-06 *Gste*](https://anopheles-genomic-surveillance.github.io/selection-atlas/agam/alert/SA-AGAM-06.html).

The **Methods** and **Glossary** pages explain the analysis. Open the Busia 2016 page now and keep
it open while you work through the notebook.

## Load the signal table

The atlas has no Python API: everything is in one table with one row per signal.

In [5]:
signals = pd.read_csv(SIGNALS_URL)

# Parse cohort_id, e.g. UG-E_Busia_gamb_2016_Q2 -> country, district, taxon, year
parts = signals["cohort_id"].str.extract(
    r"^(?P<country>[A-Z]{2})-[^_]+_(?P<district>.+)_(?P<taxon>gamb|colu|arab|biss)_(?P<year>\d{4})"
)
signals = signals.join(parts)
signals["year"] = signals["year"].astype(int)
print(signals.shape[0], "signals in", signals["cohort_id"].nunique(), "cohorts")
signals.head()

357 signals in 55 cohorts


,cohort_id,contig,gcenter,pcenter,delta_i,stat_max,gpos_max,ppos_max,focus_gstart,focus_gstop,span1_gstart,span1_gstop,span2_gstart,span2_gstop,focus_pstart,focus_pstop,span1_pstart,span1_pstop,span2_pstart,span2_pstop,amplitude,decay,skew,decay_left,decay_right,baseline,aic,bic,rss,constant_aic,country,district,taxon,year
0,BF-02_Comoe_colu_2011,2RL,49.774966,24887483,1114,0.192901,49.782798,24891399,49.729469,49.797714,49.592979,49.865959,49.410993,49.956952,24864735,24898857,24796490,24932980,24705497,24978476,0.098498,0.128684,0.500000,0.181986,0.090993,0.040685,-16087.112487,-16059.775632,0.177111,-14972.683432,BF,Comoe,colu,2011
1,BF-02_Comoe_colu_2011,2RL,57.008788,28504394,3039,0.217593,56.897334,28448667,56.868656,57.167612,56.448261,57.644083,55.887735,58.279379,28434329,28583806,28224131,28822042,27943868,29139690,0.137147,0.596741,-0.090323,0.560526,0.635296,0.042323,-14812.279513,-14785.128942,0.256334,-11772.542226,BF,Comoe,colu,2011
2,BF-02_Comoe_colu_2011,2RL,121.093838,64317550,1806,0.745370,121.201260,64371261,120.772048,121.254733,119.806680,121.737417,118.519521,122.380996,64156656,64397998,62681493,64639340,60107177,64961130,0.601599,0.910158,0.500000,1.287158,0.643579,0.030378,-6741.255204,-6715.564839,5.904284,-4934.948152,BF,Comoe,colu,2011
3,BF-02_Comoe_colu_2011,2RL,166.400346,86970804,3144,0.455247,166.381782,86961522,166.343604,166.463341,166.173379,166.652329,165.946413,166.904312,86942434,87002302,86857321,87096796,86743838,87222787,0.347096,0.239148,-0.075424,0.226966,0.251983,0.037810,-12868.134181,-12841.140249,0.617196,-9723.444518,BF,Comoe,colu,2011
4,BF-02_Comoe_colu_2011,2RL,166.400397,86970830,3144,0.455247,166.381782,86961522,166.343429,166.463575,166.172524,166.653108,165.944652,166.905819,86942346,87002419,86856894,87097186,86742957,87223541,0.347023,0.239971,-0.074630,0.227873,0.252711,0.037561,-12850.256117,-12823.268309,0.617240,-9706.156979,BF,Comoe,colu,2011


### East African cohorts

We keep cohorts from Uganda, Kenya and Tanzania. Muleba and Tarime (Tanzania) are on the southern
and eastern shores of Lake Victoria.

In [6]:
EAST_AFRICA = ["UG", "KE", "TZ"]
ea = signals.query("country in @EAST_AFRICA")
ea.groupby(["country", "cohort_id"]).size().rename("n_signals").to_frame()

n_signals
country cohort_id                             
TZ      TZ-05_Muleba_arab_2015_Q1            2
        TZ-05_Muleba_arab_2015_Q2            3
        TZ-05_Muleba_gamb_2015_Q2            2
        TZ-13_Tarime_arab_2012_Q3            3
        TZ-26_Moshi_arab_2012_Q3             3
UG      UG-C_Kalangala_gamb_2015_Q2          6
        UG-E_Busia_gamb_2016_Q2              4
        UG-E_Mayuge_gamb_2017_Q2             3
        UG-E_Tororo_arab_2012_Q4             3
        UG-E_Tororo_gamb_2012_Q4             5
        UG-W_Kanungu_gamb_2012_Q4            5

There is no Kenyan *An. gambiae* cohort in this version of the atlas. The Teso North (Busia
County, Kenya) mosquitoes we used in Module 3.1 were added to *Ag3* more recently.

## Converting between arms and compound contigs

Gene annotations use the chromosome arms (2R, 2L, 3R, 3L, X); the atlas uses 2RL and 3RL. The
offset for a position on 2L is the length of 2R, and for 3L the length of 3R. We read the arm
lengths from the reference genome rather than typing them in.

In [7]:
ARMS = ["2R", "2L", "3R", "3L", "X"]
ARM_LENGTHS = {arm: ag3.genome_sequence(region=arm).shape[0] for arm in ARMS}
ARM_LENGTHS

{'2R': 61545105, '2L': 49364325, '3R': 53200684, '3L': 41963435, 'X': 24393108}

In [8]:
# atlas contig -> (first arm, second arm); the second arm is shifted by the first arm's length
COMPOUND = {"2RL": ("2R", "2L"), "3RL": ("3R", "3L")}


def to_atlas(arm, pos):
    # Arm coordinates (e.g. 2L, 34_100_000) -> atlas coordinates (e.g. 2RL, 95_645_105).
    for compound, (first, second) in COMPOUND.items():
        if arm == first:
            return compound, pos
        if arm == second:
            return compound, pos + ARM_LENGTHS[first]
    return arm, pos  # X


def to_arm(contig, pos):
    # Atlas coordinates -> arm coordinates. Works on single positions or arrays.
    if contig not in COMPOUND:
        return contig, pos
    first, second = COMPOUND[contig]
    offset = ARM_LENGTHS[first]
    return np.where(pos > offset, second, first), np.where(pos > offset, pos - offset, pos)


to_arm("2RL", 95_646_328)

(array('2L', dtype='<U2'), array(34101223))

The Busia signal at 2RL ~95.65 Mb is on **2L at ~34.10 Mb**. `malariagen_data` knows about the
compound contigs too, so we can check our offsets against its own conversion:

In [9]:
gf_2rl = ag3.genome_features(region="2RL:95,600,000-95,700,000", attributes=["ID", "Name"]).query("type == 'gene'")
gf_2l = ag3.genome_features(region="2L:34,054,895-34,154,895", attributes=["ID", "Name"]).query("type == 'gene'")
assert list(gf_2rl["ID"]) == list(gf_2l["ID"])
assert (gf_2rl["start"].values - gf_2l["start"].values == ARM_LENGTHS["2R"]).all()
gf_2rl[["ID", "Name", "contig", "start", "end"]].head()

,ID,Name,contig,start,end
1,AGAP006540,NaN,2RL,95594665,95621447
6,AGAP006539,NaN,2RL,95609214,95610172
25,AGAP006541,NaN,2RL,95621633,95627643
34,AGAP006542,NaN,2RL,95622697,95624456
63,AGAP006543,NaN,2RL,95632535,95638911


In [10]:
signals["arm"] = signals["contig"]
signals["arm_pcenter"] = signals["pcenter"]
for compound in COMPOUND:
    loc = signals["contig"] == compound
    arm, pos = to_arm(compound, signals.loc[loc, "pcenter"].values)
    signals.loc[loc, "arm"] = arm
    signals.loc[loc, "arm_pcenter"] = pos

busia = signals.query("cohort_id == @BUSIA").sort_values("delta_i", ascending=False)
busia[["contig", "pcenter", "arm", "arm_pcenter", "stat_max", "delta_i", "focus_pstart", "focus_pstop"]]

,contig,pcenter,arm,arm_pcenter,stat_max,delta_i,focus_pstart,focus_pstop
337,2RL,28482008,2R,28482008,0.804687,3896,28392356,28562669
338,2RL,95646328,2L,34101223,0.375000,3121,95578843,95697699
339,3RL,28591758,3R,28591758,0.355035,2233,28573591,28623590
340,X,15286664,X,15286664,0.154514,1676,15177805,15399955


## What is inside the Busia signals?

Let's list the genes in the `focus` interval of each Busia 2016 signal. `genome_features()`
accepts compound contig regions directly.

In [11]:
for _, sig in busia.iterrows():
    region = f"{sig.contig}:{sig.focus_pstart}-{sig.focus_pstop}"
    genes = ag3.genome_features(region=region, attributes=["ID", "Name", "description"]).query("type == 'gene'")
    named = genes["Name"].dropna().tolist()
    print(f"{region} (arm {sig.arm}:{sig.arm_pcenter:,}) stat_max={sig.stat_max:.2f} "
          f"delta_i={sig.delta_i}: {len(genes)} genes; named: {', '.join(named[:15])}")

2RL:28392356-28562669 (arm 2R:28,482,008) stat_max=0.80 delta_i=3896: 16 genes; named: CYP6AA1, CYP6AA2, COEAE6O, CYP6P15P, CYP6P3, CYP6P5, CYP6P4, CYP6P1, CYP6P2, CYP6AD1


2RL:95578843-95697699 (arm 2L:34,101,223) stat_max=0.37 delta_i=3121: 31 genes; named: 


3RL:28573591-28623590 (arm 3R:28,591,758) stat_max=0.36 delta_i=2233: 13 genes; named: GSTE8, GSTE6, GSTE5, GSTE4, GSTE2, GSTE1, GSTE7, GSTE3


X:15177805-15399955 (arm X:15,286,664) stat_max=0.15 delta_i=1676: 14 genes; named: CYP9K1, CPR125, CASPS7, MIP


The strongest Busia signal (H12 up to 0.80) is centred on the *Cyp6aa/Cyp6p* cluster on 2R, almost
exactly on *Cyp6aa1*. This is the locus of the East African **triple mutant** [4]: the
`Cyp6aap_Dup1` duplication of *Cyp6aa1*, CYP6P4 I236M, and a transposable element (ZZB) insertion
near *Cyp6p4*. In Module 3.1 we saw the duplication in 95–100 % of Busia mosquitoes. On Day 4 you
will see that two markers of this haplotype on the Ag-vampIR panel, `Cyp6p4_I236M` and `Cyp6_tag8`,
are associated with surviving deltamethrin in Siaya. The other Busia signals sit on the *Gste*
cluster (3R), on *Cyp9k1* (X) and at ~34.1 Mb on 2L.

### Seeing the peak model

To make the intervals concrete, we redraw the fitted peak for the Busia *Cyp6* signal from its
parameters. The model is fitted in genetic coordinates (cM).

In [12]:
def peak_model(g, sig):
    # Skewed exponential peak used by the atlas (genetic coordinates).
    dist = g - sig.gcenter
    decay = np.where(dist < 0, sig.decay_left, sig.decay_right)
    return sig.baseline + sig.amplitude * np.exp(-np.abs(dist) / decay)


sig = busia.query("contig == '2RL'").iloc[0]
g = np.linspace(sig.gcenter - 3, sig.gcenter + 3, 500)
fig = go.Figure()
for name, alpha in [("span2", 0.10), ("span1", 0.20), ("focus", 0.35)]:
    fig.add_vrect(x0=sig[f"{name}_gstart"], x1=sig[f"{name}_gstop"], fillcolor="firebrick",
                  opacity=alpha, line_width=0, annotation_text=name, annotation_position="top left")
fig.add_trace(go.Scatter(x=g, y=peak_model(g, sig), name="fitted peak", line=dict(color="black")))
fig.add_hline(y=sig.baseline, line_dash="dot", annotation_text="baseline")
fig.update_layout(title=f"{BUSIA}, 2RL signal (fitted model)", xaxis_title="Genetic position (cM)",
                  yaxis_title="H12", width=750, height=380, yaxis_range=[0, 1])
fig

The observed H12 values that this curve was fitted to are shown on the cohort page of the website,
and we recompute them in Module 3.3.

## Do signals overlap your candidate genes?

We look up the candidate genes, convert their coordinates to atlas contigs, and find every signal in
every cohort whose focus, span1 or span2 overlaps each gene.

In [13]:
gf = ag3.genome_features(attributes=["ID", "Name", "description"]).query("type == 'gene'")
cand = gf.query("ID in @CANDIDATE_GENES")[["ID", "Name", "contig", "start", "end"]].copy()
atlas_coords = [to_atlas(r.contig, r.start) + to_atlas(r.contig, r.end)[1:] for r in cand.itertuples()]
cand[["atlas_contig", "atlas_start", "atlas_end"]] = pd.DataFrame(atlas_coords, index=cand.index)
cand

,ID,Name,contig,start,end,atlas_contig,atlas_start,atlas_end
73275,AGAP002862,CYP6AA1,2R,28480576,28482637,2RL,28480576,28482637
155971,AGAP009193,GSTE4,3R,28595948,28596868,3RL,28595948,28596868


In [14]:
def signals_at(atlas_contig, start, end):
    # All signals whose span2 overlaps [start, end], with the narrowest interval that overlaps.
    df = signals.query(
        "contig == @atlas_contig and span2_pstart <= @end and span2_pstop >= @start"
    ).copy()
    df["interval"] = "span2"
    for name in ["span1", "focus"]:  # later (narrower) intervals overwrite
        hit = (df[f"{name}_pstart"] <= end) & (df[f"{name}_pstop"] >= start)
        df.loc[hit, "interval"] = name
    df["distance_to_centre"] = np.where(
        df["pcenter"] < start, start - df["pcenter"],
        np.where(df["pcenter"] > end, df["pcenter"] - end, 0),
    )
    return df


hits = pd.concat(
    [signals_at(r.atlas_contig, r.atlas_start, r.atlas_end).assign(gene=r.ID, gene_name=r.Name)
     for r in cand.itertuples()]
)
cols = ["gene", "gene_name", "cohort_id", "interval", "distance_to_centre", "stat_max", "delta_i"]
hits.query("country in @EAST_AFRICA")[cols].sort_values(["gene", "cohort_id"])

,gene,gene_name,cohort_id,interval,distance_to_centre,stat_max,delta_i
319,AGAP002862,CYP6AA1,TZ-05_Muleba_arab_2015_Q1,focus,41375,0.925049,3587
322,AGAP002862,CYP6AA1,TZ-05_Muleba_arab_2015_Q2,focus,46718,0.920242,4632
323,AGAP002862,CYP6AA1,TZ-05_Muleba_gamb_2015_Q2,focus,7439,0.745370,2524
326,AGAP002862,CYP6AA1,TZ-13_Tarime_arab_2012_Q3,focus,62144,0.837936,3923
328,AGAP002862,CYP6AA1,TZ-26_Moshi_arab_2012_Q3,focus,20893,0.761999,2793
331,AGAP002862,CYP6AA1,UG-C_Kalangala_gamb_2015_Q2,focus,0,0.615833,7784
337,AGAP002862,CYP6AA1,UG-E_Busia_gamb_2016_Q2,focus,0,0.804687,3896
341,AGAP002862,CYP6AA1,UG-E_Mayuge_gamb_2017_Q2,focus,37552,0.701814,2292
344,AGAP002862,CYP6AA1,UG-E_Tororo_arab_2012_Q4,focus,117155,0.676116,3122
347,AGAP002862,CYP6AA1,UG-E_Tororo_gamb_2012_Q4,focus,11823,0.699800,5965


Both default genes are inside the **focus** of a Busia 2016 signal: CYP6AA1 contains the fitted
centre of the 2R signal, and GSTE4 is a few kb from the centre of the *Gste* signal. Being in the
focus is suggestive, not proof: the *Gste* focus contains several GST genes, and the *Cyp6* focus
spans about 170 kb and many genes. Which variant was selected needs the follow-up in Module 3.3.

### How widespread is selection at your genes?

Counting cohorts with a signal at the gene across the whole atlas shows whether the locus is under
selection in many places and taxa (parallel selection) or only locally.

In [15]:
summary = (
    hits.query("interval in ['focus', 'span1']")
    .groupby(["gene_name", "country", "taxon"])["cohort_id"].nunique()
    .rename("n_cohorts").reset_index()
)
px.bar(summary, x="country", y="n_cohorts", color="taxon", facet_col="gene_name",
       title="Cohorts with a signal (focus or span1) at each candidate gene", height=400, width=900)

## Comparing cohorts around Lake Victoria

A lane plot shows each cohort's signals at a locus as nested bars (thin = span2, medium = span1,
thick = focus), with a dot at the centre coloured by `stat_max`. Vertical lines mark genes of
interest. We plot in arm coordinates.

In [16]:
def plot_lanes(atlas_contig, start, end, cohort_ids, mark_genes=(), title=None):
    df = signals.query(
        "contig == @atlas_contig and cohort_id in @cohort_ids "
        "and span2_pstart <= @end and span2_pstop >= @start"
    ).sort_values("year")
    arm = str(to_arm(atlas_contig, (start + end) // 2)[0])
    conv = lambda p: to_arm(atlas_contig, np.asarray(p))[1] / 1e6
    fig = go.Figure()
    for name, width in [("span2", 3), ("span1", 8), ("focus", 16)]:
        for r in df.itertuples():
            fig.add_trace(go.Scatter(
                x=conv([getattr(r, f"{name}_pstart"), getattr(r, f"{name}_pstop")]),
                y=[r.cohort_id] * 2, mode="lines", line=dict(width=width, color="lightcoral"),
                showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(
        x=conv(df["pcenter"]), y=df["cohort_id"], mode="markers",
        marker=dict(size=12, color=df["stat_max"], colorscale="Reds", cmin=0, cmax=1,
                    colorbar=dict(title="stat_max"), line=dict(color="black", width=1)),
        text=[f"delta_i={d}" for d in df["delta_i"]], showlegend=False))
    for gene_id in mark_genes:
        g = gf.query("ID == @gene_id").iloc[0]
        fig.add_vline(x=(g.start + g.end) / 2e6, line_dash="dot",
                      annotation_text=g.Name if isinstance(g.Name, str) else gene_id)
    fig.update_layout(title=title, xaxis_title=f"{arm} position (Mb)", width=900,
                      height=150 + 35 * df["cohort_id"].nunique(),
                      xaxis_range=[conv(start), conv(end)])
    return fig


UG_TZ = sorted(ea["cohort_id"].unique())
plot_lanes("2RL", 27_000_000, 30_000_000, UG_TZ, mark_genes=["AGAP002862", "AGAP002894"],
           title="Signals at the Cyp6aa/Cyp6p cluster, East Africa")

Every East African cohort, including the *An. arabiensis* cohorts, has a signal at the *Cyp6*
cluster. In the *An. gambiae* cohorts the centres fall within a few tens of kb of *Cyp6aa1*.
Now the *Gste* cluster on 3R:

In [17]:
plot_lanes("3RL", 28_000_000, 29_200_000, UG_TZ, mark_genes=["AGAP009193", "AGAP009194"],
           title="Signals at the Gste cluster, East Africa")

### Signals through time

Uganda has *An. gambiae* cohorts from 2012 to 2017. Below, for each Ugandan cohort we take the
strongest signal at four loci and plot its `stat_max` by year.

In [18]:
LOCI = {  # name: (atlas contig, start, end) in atlas coordinates
    "Cyp6aa/Cyp6p": ("2RL", 28_480_000, 28_510_000),
    "Gste": ("3RL", 28_590_000, 28_605_000),
    "Cyp9k1": ("X", 15_240_000, 15_245_000),
    "2L ~34.1 Mb": ("2RL", ARM_LENGTHS["2R"] + 34_050_000, ARM_LENGTHS["2R"] + 34_150_000),
}
rows = []
for locus, (c, a, b) in LOCI.items():
    df = signals_at(c, a, b).query("country == 'UG' and taxon == 'gamb' and interval != 'span2'")
    best = df.sort_values("stat_max").groupby("cohort_id").tail(1)
    rows.append(best.assign(locus=locus))
trend = pd.concat(rows)
px.scatter(trend, x="year", y="stat_max", color="locus", symbol="district",
           hover_data=["cohort_id", "delta_i"], range_y=[0, 1],
           title="Strength of signals at four loci in Ugandan An. gambiae", width=800, height=420)

Read trends like this with care. The cohorts come from different districts, the sample sizes and
window sizes differ, and H12 is not an allele frequency: a sweep can become *less* visible to H12 as
it approaches fixation and recombination erodes the haplotype, or if several haplotypes are
selected (a soft sweep). Frequencies of the selected variants (Module 3.1) are the better way to
track change over time.

### Exercise 1

For each of your own candidate genes, which Busia 2016 signal (if any) contains it, and in which
interval? How far is it from the signal centre? How many genes are in that signal's focus?

<details><summary>Show answer</summary>

Use `hits.query("cohort_id == @BUSIA")` for the interval and distance, and the gene listing loop
above for the focus contents. With the defaults: CYP6AA1 contains the centre of the 2R signal
(distance 0), and GSTE4 lies a few kb from the centre of the 3R *Gste* signal, inside its focus.
A gene 500 kb from the centre and only in span2 is much weaker evidence than a gene at the centre.

</details>

### Exercise 2

Tororo, Kanungu and Kalangala have a strong signal on 2RL at ~64.3 Mb. Convert it to arm
coordinates, list the genes in its span1 and name the best-known one. Busia 2016 has no signal
there: suggest two reasons.

<details><summary>Show answer</summary>

```python
vgsc = signals.query("contig == '2RL' and 63_000_000 < pcenter < 66_000_000")
vgsc[["cohort_id", "arm", "arm_pcenter", "stat_max", "span1_pstart", "span1_pstop"]]
```
The centres are on 2L at ~2.72–2.85 Mb, and span1 includes *Vgsc* (AGAP004707, 2L:2,358,158–2,431,617),
the pyrethroid target site (kdr). Across the whole atlas, signals here centre ~0.3 Mb to the
right of *Vgsc* and the focus usually misses the gene: a reminder that the centre is not always
on the selected gene (skew, recombination rate and hitchhiking all shift it). Busia may lack a signal because kdr is near
fixation on several haplotypes (a soft or old sweep that H12 with 24 mosquitoes does not detect
as a peak), or simply because of the small sample; check kdr frequencies (Module 2.2) and the
Busia page on the website.

</details>

### Exercise 3

Across the whole atlas, how many cohorts, countries and taxa have a signal whose **focus**
overlaps *Gste2* (AGAP009194)? Is the *Gste* locus under selection only in East Africa?

<details><summary>Show answer</summary>

```python
g = gf.query("ID == 'AGAP009194'").iloc[0]
c, a = to_atlas(g.contig, g.start)
gste = signals_at(c, a, a + (g.end - g.start)).query("interval == 'focus'")
gste["cohort_id"].nunique(), gste["country"].nunique(), gste["taxon"].unique()
```
31 cohorts from 9 countries (Burkina Faso, Benin, DRC, CAR, Côte d'Ivoire, Cameroon, Ghana, Mali
and Uganda), 19 of them *An. coluzzii* and 12 *An. gambiae*. Selection at *Gste* is widespread
across West, Central and East Africa, not a local Busia phenomenon.

</details>

### Exercise 4

Rank the four Busia 2016 signals by `delta_i` and by `stat_max`. Do the rankings agree? What does
each column tell you?

<details><summary>Show answer</summary>

Both rank the *Cyp6* signal first (ΔAIC ~3,900, H12 0.80), then 2L ~34.1 Mb, then *Gste*, then
*Cyp9k1* (ΔAIC ~1,700, H12 0.15). `delta_i` measures how clearly a peak shape stands out from a
flat background; `stat_max` measures how much haplotype homozygosity there is at the top of the
peak (roughly, how far the sweep has gone). A broad, low peak can have a high ΔAIC.

</details>

### Exercise 5

On the website, open the [Busia 2016 cohort page](https://anopheles-genomic-surveillance.github.io/selection-atlas/agam/cohort/UG-E_Busia_gamb_2016_Q2.html)
and the [*Cyp9k1* alert](https://anopheles-genomic-surveillance.github.io/selection-atlas/agam/alert/SA-AGAM-05.html).
On the X chromosome, do H12, G123 and iHS agree about the *Cyp9k1* signal? Which other East African
cohorts have a *Cyp9k1* signal?

<details><summary>Show answer</summary>

The Busia *Cyp9k1* signal is weak in H12 (max ~0.15). That fits Module 3.1: the amplified
*Cyp9k1* haplotype is carried by only about a third of Busia mosquitoes, an incomplete sweep.
iHS is the statistic most sensitive to incomplete sweeps, so check whether the iHS track shows
the locus more clearly than H12 and G123. The alert page (or
`hits`-style queries on X:15.24 Mb) lists Kanungu, Kalangala, Tororo and Mayuge as other East African
cohorts with signals near *Cyp9k1*; Kanungu's is the strongest (H12 ~0.48), matching the high
*Cyp9k1_Dup8* frequency there.

</details>

## Quiz

Check your understanding. The quiz runs in Colab and in the course book.

In [ ]:
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
display_quiz("https://raw.githubusercontent.com/sanjaynagi/anopheles-omics-training/main/quizzes/3-2-selection-atlas.json")

## Summary

- H12, G123 and iHS detect the haplotype footprints of recent selection; the atlas calls signals
  from H12 by fitting a skewed exponential peak and keeping peaks with ΔAIC > 1000 and H12 > 0.1.
- `focus`, `span1` and `span2` are nested intervals around the fitted centre; use focus for the
  likely target and span2 for the broad footprint.
- The atlas scans compound contigs: 2RL = 2R then 2L, 3RL = 3R then 3L; convert with the arm lengths.
- Busia 2016 has four signals: *Cyp6aa/Cyp6p* (the triple-mutant locus, the strongest), 2L ~34.1 Mb,
  *Gste* and *Cyp9k1*. Every East African cohort has a signal at the *Cyp6* cluster; the *Gste*
  signal is shared with Tororo and Kanungu.

## Well done!

Add the selection evidence for your genes to your dossier. In the optional Module 3.3 we zoom in on
the Busia *Cyp6* sweep, find the swept haplotypes and choose SNPs that tag them for the assays on Day 4.

## References

1. Garud NR, Messer PW, Buzbas EO, Petrov DA (2015). Recent selective sweeps in North American *Drosophila melanogaster* show signatures of soft sweeps. *PLoS Genetics* 11:e1005004. https://doi.org/10.1371/journal.pgen.1005004
2. Harris AM, Garud NR, DeGiorgio M (2018). Detection and classification of hard and soft sweeps from unphased genotypes by multilocus genotype identity. *Genetics* 210:1429–1452. https://doi.org/10.1534/genetics.118.301502
3. Voight BF, Kudaravalli S, Wen X, Pritchard JK (2006). A map of recent positive selection in the human genome. *PLoS Biology* 4:e72. https://doi.org/10.1371/journal.pbio.0040072
4. Njoroge H *et al.* (2022). Identification of a rapidly-spreading triple mutant for high-level metabolic insecticide resistance in *Anopheles gambiae* provides a real-time molecular diagnostic for antimalarial intervention deployment. *Molecular Ecology*. https://doi.org/10.1111/mec.16591
5. Nagi SC, Harding NJ, Lawniczak MKN, Donnelly MJ, Miles A. An atlas of positive selection in the genomes of major malaria vectors (in preparation). Website: https://anopheles-genomic-surveillance.github.io/selection-atlas/
6. The *Anopheles gambiae* 1000 Genomes Consortium (2017). Genetic diversity of the African malaria vector *Anopheles gambiae*. *Nature* 552:96–100. https://doi.org/10.1038/nature24995